# StoryPix — free notebook GPU (batch mode)

StoryPix is local-first: it normally runs on your own computer. This notebook is the optional **free-GPU path**
(Google Colab / Kaggle T4) for machines without a capable GPU or Apple-silicon chip. It runs the **same pipeline**
as the app — story analysis with the trained models → character anchors → SDXL + IP-Adapter scenes → evaluation —
in batch mode and writes the storybook (PNG images, PDF, ZIP) to the notebook's disk.

* No web server and no tunnel: free notebook tiers are for interactive use, so this notebook simply produces files.
* Sessions are time-limited and a GPU is not always available on free tiers.
* First run downloads ~12 GB of model weights into the session (several minutes on Colab's network).

Runtime → Change runtime type → **T4 GPU**, then run the cells in order.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone --depth 1 https://github.com/bajpaisushil/storypics.git
%cd storypics/backend
!pip install -q -r requirements-colab.txt

In [ ]:
# ~12 GB of open weights: SDXL base (OpenRAIL++), SDXL-Lightning LoRA (OpenRAIL++), IP-Adapter Plus (Apache-2.0),
# fp16-fix VAE (MIT), SigLIP + DINOv2 + MiniLM encoders (Apache-2.0). Licences: docs/ARCHITECTURE.md
!python scripts/download_models.py

In [ ]:
story = """Once upon a time, there was a hen who lived on a farm. One day, she met a cow.
They became good friends and explored the farm together. One evening, they helped a lost little rabbit find its home."""
open("story.txt", "w").write(story)
STYLE = "storybook"   # cartoon | realistic | anime | watercolor | 3d | storybook | digital_painting | comic
SCENES = "auto"       # or 3, 5, 8, 10

In [ ]:
# On a CUDA GPU the default profile is "quality" (SDXL, 28 steps, CFG + negative prompts). For faster drafts:
# import os; os.environ["STORYPIX_PROFILE"] = "lightning"
!python -m app.cli analyze story.txt --scenes {SCENES}
!python -m app.cli generate story.txt --style {STYLE} --scenes {SCENES} --aspect 4:3 --out /content/out

In [ ]:
import glob
from IPython.display import Image, display
for path in sorted(glob.glob("/content/out/data/media/*/scene_*_v1.png")):
    print(path.rsplit("/", 1)[-1]); display(Image(path, width=640))

In [ ]:
# Download the results (Colab). On Kaggle, use the Output panel instead.
try:
    from google.colab import files
    files.download("/content/out/storybook.pdf"); files.download("/content/out/storybook.zip")
except ImportError:
    print("Files are in /content/out")

## Optional: retrain the custom models on the notebook

The four StoryPix models are small and train in minutes even on a laptop, but you can reproduce them here too:

```
!python -m training.datasets.download_data text && python -m training.datasets.download_data images
!python -m training.datasets.prepare_scene_data && python -m training.train_scene_segmenter
!python -m training.datasets.prepare_tagger_data && python -m training.train_character_model
!python -m training.train_image_models extract && python -m training.train_image_models train
!for t in scene tagger alignment quality; do python -m training.evaluate $t; done
```